Script was not needed; cloud training with some tricks worked without needing to throw away data 

In [1]:
import pandas as pd
import os
import sys
sys.path.append('../utils')
from  configloader import Configloader
from database import Database


In [ ]:
config = Configloader()
basedir = config.get('settings', 'image_directory')
augment_csv_dump = os.path.join(basedir, 'CSV-data', 'brand phase')

db = Database(config)
db.connect()

Connection established


In [3]:
testdata = pd.read_csv(os.path.join(augment_csv_dump, 'testdata_brandphase.csv'))

In [4]:
len(testdata)

798122

In [5]:
def shrinkdf(df, cols, minbound):
    return df.groupby(cols).head(minbound).reset_index(drop=True)


In [6]:
query = """SELECT 
           brand, model, year, shelltype, angletag_predicts.model_label, images.id
        FROM 
            listings
        JOIN images ON images.listing_id = listings.id
        JOIN angletag_predicts ON angletag_predicts.image_id = images.id"""

sqldata = pd.DataFrame(db.execute_query(query))

In [7]:
sqldata.sample(3)

,brand,model,year,shelltype,model_label,id
4778476,ford,Focus,2002,Break,frontleft,5029098
6437283,renault,Captur,2023,SUV/4x4/Pick-up,frontleft,7098496
11497521,volkswagen,Polo (alle),2016,Stadswagen,frontleft,12158734


In [8]:
testdata.sample(5)

,image_id,model_label,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand,abs_path
76908,2498198,rearleft,1.000000,73,134,642,509,2498198,0.949114,0.995570,0.853385,0.743426,volvo,/home/frederic/Documents/automotive_image_data...
650229,4031771,front,1.000000,68,62,593,522,4031771,0.993907,0.999999,0.999180,0.999180,volkswagen,/home/frederic/Documents/automotive_image_data...
657988,6837553,frontleft,1.000000,11,129,650,513,6837553,0.999780,0.999925,0.993030,0.998216,nissan,/home/frederic/Documents/automotive_image_data...
48422,15236064,frontleft,1.000000,73,98,679,492,15236064,0.987614,0.999989,0.998846,0.999573,toyota,/home/frederic/Documents/automotive_image_data...
385564,2705230,rearright,0.999545,46,37,704,508,2705230,0.997577,0.999806,0.971014,0.980025,hyundai,/home/frederic/Documents/automotive_image_data...


In [9]:
merged_testdata = sqldata.merge(testdata, left_on='id', right_on='image_id', how='right')

In [10]:
merged_testdata.sample(5)

,brand_x,model,year,shelltype,model_label_x,id,image_id,model_label_y,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand_y,abs_path
669910,audi,A6,2017,Break,front,3252046,3252046,front,0.999997,-1,-1,-1,-1,3252046,0.954785,0.999527,0.967634,0.959697,audi,/home/frederic/Documents/automotive_image_data...
348276,mercedes-benz,E-Klasse (alle),2008,Berline,frontright,13296030,13296030,frontright,1.000000,130,226,680,510,13296030,0.995046,0.973231,0.504948,0.740485,mercedes-benz,/home/frederic/Documents/automotive_image_data...
536398,hyundai,KONA,2021,SUV/4x4/Pick-up,rearright,2816214,2816214,rearright,1.000000,22,71,726,450,2816214,0.999128,0.999999,0.994134,0.999049,hyundai,/home/frederic/Documents/automotive_image_data...
246498,renault,Twingo,2018,Stadswagen,right,13955975,13955975,right,1.000000,11,145,721,459,13955975,0.971343,0.999998,0.996785,0.998108,renault,/home/frederic/Documents/automotive_image_data...
443205,volvo,XC60,2023,SUV/4x4/Pick-up,frontleft,11321246,11321246,frontleft,0.999885,64,0,748,426,11321246,0.245912,0.999676,0.698044,0.995969,volvo,/home/frederic/Documents/automotive_image_data...


In [11]:
len(merged_testdata)

798122

In [12]:
len(testdata)

798122

In [13]:
merged_testdata.head(3)

,brand_x,model,year,shelltype,model_label_x,id,image_id,model_label_y,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand_y,abs_path
0,kia,Sportage,2020,SUV/4x4/Pick-up,front,8438302,8438302,front,1.000000,148,111,640,529,8438302,0.291355,0.993442,0.932474,0.432448,kia,/home/frederic/Documents/automotive_image_data...
1,ford,Transit (alle),2024,Bestelwagen,frontleft,12835404,12835404,frontleft,1.000000,13,85,741,463,12835404,0.737102,0.998978,0.798775,0.990299,ford,/home/frederic/Documents/automotive_image_data...
2,opel,Zafira Tourer,2016,Monovolume,rearleft,14772480,14772480,rearleft,0.757401,56,64,727,420,14772480,0.529626,0.999941,0.993007,0.994510,opel,/home/frederic/Documents/automotive_image_data...


In [14]:
reduced = shrinkdf(merged_testdata, ['brand_x', 'model_label_x', 'year', 'model', 'shelltype'],4)

In [15]:
merged_testdata.model_label_x.value_counts()

model_label_x
frontleft     164254
rearright     114181
frontright    112834
rearleft      100683
front          97459
rear           76241
left           72202
right          60268
Name: count, dtype: int64

In [16]:
reduced.model_label_x.value_counts()

model_label_x
frontleft     41816
frontright    37445
rearright     34854
front         33963
rearleft      33787
rear          28806
left          27492
right         25221
Name: count, dtype: int64

In [17]:
def reducer(df, max_samples, by):
    """Training gets stuck when using the testdata during the traininprocess
    as the models get more complex; in stead of using the full testset, this 
    function will subsample it in smaller sections using a random selection
    It'll keep procentually more of smaller 'by' values 
    
    """
    sampled = []
    for _, group in df.groupby(by):
        if max_samples > len(group):
            samplesize = len(group)
        else:
            samplesize = max_samples
        small_df = group.sample(samplesize)
        sampled.append(small_df)
    return pd.concat(sampled).reset_index(drop=True)

reduced2 = reducer(testdata.query('model_label=="front"'), 1000, 'brand')

In [18]:
x='brand'
reduced2[x].nunique()

30